# Notebook 37 — Tap distribution at fixed weight count: do live channels mediate collapse?

Notebook 36 found that live input channels and surviving input weights predict collapse equally well,
because magnitude pruning distributes survivors almost exactly as random survival would, making the two
quantities collinear. This notebook breaks the collinearity by constructing the first-layer mask by hand.

**Design.** On the five paired CNN baselines, `conv.3` and `head` are pruned at 80% by magnitude as before,
while `conv.0` (64 filters x 3 taps) receives a hand-built mask at a fixed surviving-tap count:
- 38 taps, **spread**: the 38 filters with the largest peak tap each keep their single largest tap (38 live filters);
- 38 taps, **concentrated**: the 13 filters with the largest total magnitude keep all taps, 12 full and one
  with two taps (13 live filters);
- 96 taps, **spread**: every filter keeps its largest tap, then the 32 largest remaining taps are added (64 live);
- 96 taps, **concentrated**: the 32 filters with the largest total magnitude keep all three taps (32 live).
Magnitude-pruned references at the same tap counts come from Notebook 35 (38 taps: ~35 live; 96 taps:
~58 live). Live-filter counts are asserted after fine-tuning.

**Gate (stated before running).** Live channels mediate collapse if, at 38 taps, the concentrated mask
loses at least 0.10 more macro-F1 than the spread mask, and the ordering (concentrated worse) also holds
at 96 taps. If spread and concentrated masks collapse alike at equal tap counts, the law stays at the
surviving-weight level. Resumable per (seed, condition). GPU runtime required.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
CONDITIONS = [('tapspread38', 38, 'spread'), ('tapconc38', 38, 'concentrated'), ('tapspread96', 96, 'spread'), ('tapconc96', 96, 'concentrated')]

ARCH = 'cnn1d'; ARCH_KW = {'channels': (64, 128)}; BASE_CELL = 'M0_paired'
print('conditions:', [c[0] for c in CONDITIONS])

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)

# Hand-built conv.0 masks at a fixed surviving-tap count
def conv0_mask(w, n_taps, mode):
    # w: (64, 1, 3) conv.0 weight; returns a bool mask with exactly n_taps True entries
    F, _, K = w.shape; a = w.abs().reshape(F, K); mask = torch.zeros(F, K, dtype=torch.bool)
    if mode == 'spread':
        top_tap = a.argmax(dim=1)                                   # each filter's largest tap
        if n_taps <= F:
            order = a.max(dim=1).values.argsort(descending=True)[:n_taps]
            mask[order, top_tap[order]] = True                       # n_taps filters x 1 tap
        else:
            mask[torch.arange(F), top_tap] = True                    # every filter x 1 tap (F live)
            rem = a.clone(); rem[mask] = -1.0
            idx = rem.reshape(-1).argsort(descending=True)[:n_taps - F]
            mask.reshape(-1)[idx] = True
    elif mode == 'concentrated':
        order = a.sum(dim=1).argsort(descending=True)                # filters by total magnitude
        full, extra = divmod(n_taps, K)
        mask[order[:full], :] = True                                 # full filters
        if extra:
            f = order[full]; mask[f, a[f].argsort(descending=True)[:extra]] = True
    else:
        raise ValueError(mode)
    assert int(mask.sum()) == n_taps, (int(mask.sum()), n_taps)
    return mask.reshape(F, 1, K)

def apply_condition(model, n_taps, mode):
    m = copy.deepcopy(model); names = layer_names(m)
    conv0 = [mod for mod, _ in prunable(m) if names[mod] == 'conv.0'][0]
    with torch.no_grad():
        mask = conv0_mask(conv0.weight.detach().cpu(), n_taps, mode).to(conv0.weight.device)
        conv0.weight.mul_(mask.float())
    for mod, name in prunable(m):
        if names[mod] == 'conv.0': continue
        prune.l1_unstructured(mod, name=name, amount=0.8); prune.remove(mod, name)
    return m

def live_filters(model):
    names = layer_names(model)
    conv0 = [mod for mod, _ in prunable(model) if names[mod] == 'conv.0'][0]
    nz = (conv0.weight.detach() != 0).reshape(conv0.weight.shape[0], -1)
    return int(nz.any(dim=1).sum()), int(nz.sum())

m_chk, _, _, _ = load_anchor(DATASET, ARCH, BASE_CELL, ANCHOR, arch_kwargs=ARCH_KW)
got = [layer_names(m_chk)[mod] for mod, _ in prunable(m_chk)]; assert got == ['conv.0', 'conv.3', 'head'], got
w0 = [mod for mod, _ in prunable(m_chk) if layer_names(m_chk)[mod] == 'conv.0'][0].weight.detach().cpu()
for cell, n, mode in CONDITIONS:
    mk = conv0_mask(w0, n, mode); print(f'  {cell:12s} taps {int(mk.sum()):3d}  live filters {int(mk.reshape(64, -1).any(dim=1).sum()):2d}')

In [ ]:
# Run the four conditions on five paired baselines, with resume
baseline_val, baseline_test, comp_test, macro, live_rows = {}, {}, {}, [], []
for seed in SEEDS:
    print(f'\n===== seed {seed} =====')
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, BASE_CELL, seed, arch_kwargs=ARCH_KW)
    yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val')
    yt, pt, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
    baseline_val[seed] = per_class_recall_table(yv, pv, le).set_index('label')['recall']
    baseline_test[seed] = per_class_recall_table(yt, pt, le).set_index('label')['recall']
    macro.append({'seed': seed, 'cell': 'M0', 'taps': 192, 'mode': 'none', 'test_macro_f1': f1_score(yt, pt, average='macro')})
    for cell, n_taps, mode in CONDITIONS:
        p_c = PATHS.model(DATASET, ARCH, f'{cell}_paired', seed)
        if os.path.exists(p_c):
            mp = M.build(ARCH, len(feat_cols), len(le.classes_), **ARCH_KW).to(DEVICE)
            mp.load_state_dict(torch.load(p_c, map_location=DEVICE, weights_only=False)['state_dict']); mp.eval(); print(f'  loaded {cell}')
        else:
            mp, _, _ = finetune_masked(apply_condition(m0, n_taps, mode), seed); save_ckpt(mp, le, scaler, p_c); print(f'  saved {cell}')
        live, taps = live_filters(mp)
        assert taps == n_taps, (cell, seed, taps, n_taps)
        live_rows.append({'seed': seed, 'cell': cell, 'taps': taps, 'mode': mode, 'live_filters': live, **layer_sparsity(mp)})
        yt, pc, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test')
        comp_test[(seed, cell)] = per_class_recall_table(yt, pc, le).set_index('label')['recall']
        macro.append({'seed': seed, 'cell': cell, 'taps': n_taps, 'mode': mode, 'test_macro_f1': f1_score(yt, pc, average='macro')})
print('\nall conditions complete')

In [ ]:
# Aggregate + gate
tiers = assign_validation_tiers(pd.DataFrame(baseline_val))
rows = []
for (seed, cell), rc in comp_test.items():
    r0 = baseline_test[seed]
    for cls in r0.index.intersection(rc.index):
        loss = float(r0.loc[cls] - rc.loc[cls]); band = float(tiers.loc[cls, 'validation_2sd_band'])
        rows.append({'seed': seed, 'cell': cell, 'class': cls, 'M0_test_recall': float(r0.loc[cls]), 'compressed_test_recall': float(rc.loc[cls]),
                     'recall_loss': loss, 'material_and_beyond_band': bool((loss > band) and (loss >= PRACTICAL_LOSS))})
eff = pd.DataFrame(rows); assert len(eff) > 0, 'no rows: run the condition cell in this session first'
eff.to_csv(OUT / 'tap_distribution_per_class_effects.csv', index=False)
summ = eff.groupby(['cell', 'class']).agg(mean_recall_loss=('recall_loss', 'mean'), affected_frequency=('material_and_beyond_band', 'mean')).reset_index()
summ.to_csv(OUT / 'tap_distribution_per_class_summary.csv', index=False)
mdf = pd.DataFrame(macro); mdf.to_csv(OUT / 'tap_distribution_macro_f1_wide.csv', index=False)
ldf = pd.DataFrame(live_rows); ldf.to_csv(OUT / 'tap_distribution_live_filters.csv', index=False)
m0m = mdf[mdf.cell == 'M0'].test_macro_f1.mean()

ref = pd.read_csv(OUT / 'input_starvation_dose_response.csv'); ref = ref[ref.arch == 'cnn1d']
lc = pd.read_csv(OUT / 'live_channel_per_run.csv'); lc = lc[lc.arch == 'cnn1d']
tab = []
for cell, n_taps, mode in CONDITIONS:
    g = mdf[mdf.cell == cell].test_macro_f1
    tab.append({'condition': cell, 'taps': n_taps, 'mask': mode, 'live_filters': float(ldf[ldf.cell == cell].live_filters.mean()),
                'mean_macro_f1': g.mean(), 'sd': g.std(), 'mean_macro_f1_loss': m0m - g.mean(),
                'classes_affected_ge3of5': int((summ[summ.cell == cell].affected_frequency >= 0.6).sum())})
for n_taps in (38, 96):
    r = ref[ref.surviving_input_weights == n_taps].iloc[0]
    tab.append({'condition': f'magnitude_ref_{n_taps}', 'taps': n_taps, 'mask': 'magnitude (NB35)',
                'live_filters': float(lc[lc.surviving_input_weights == n_taps].live_channels.mean()),
                'mean_macro_f1': float(r.mean_macro_f1), 'sd': float(r.sd_macro_f1), 'mean_macro_f1_loss': float(r.mean_macro_f1_loss),
                'classes_affected_ge3of5': int(r.classes_affected_ge3of5)})
tab = pd.DataFrame(tab).sort_values(['taps', 'live_filters']); tab.to_csv(OUT / 'tap_distribution_comparison.csv', index=False)
print(f'M0 five-seed macro-F1: {m0m:.4f}\n'); print(tab.round(4).to_string(index=False))

def L(c): return float(tab[tab.condition == c].mean_macro_f1_loss.iloc[0])
d38 = L('tapconc38') - L('tapspread38'); d96 = L('tapconc96') - L('tapspread96')
verdict = pd.DataFrame([
 {'criterion': 'at_38_taps_concentrated_minus_spread_loss_ge_0.10', 'value': round(d38, 4), 'pass': bool(d38 >= 0.10)},
 {'criterion': 'at_96_taps_concentrated_worse_than_spread', 'value': round(d96, 4), 'pass': bool(d96 > 0)},
])
print(); print(verdict.to_string(index=False))
print('\nLive channels mediate collapse at fixed weight count:', bool(verdict['pass'].all()))
verdict.to_csv(OUT / 'tap_distribution_gate_verdict.csv', index=False)
write_json(OUT / 'tap_distribution_environment.json', {'conditions': CONDITIONS, 'seeds': SEEDS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, this notebook's own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/37_tap_distribution_live_channel_mediation.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/tap_distribution_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 37: tap distribution at fixed surviving-tap count - spread vs concentrated conv.0 masks; live-channel mediation gate'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)